In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
#warnings 제거 및 한글 폰트 추가 가능하게
warnings.filterwarnings('ignore')
plt.rc('font', family='Malgun Gothic')

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('EternalReturn_kakaogames_2024.csv')

In [4]:
# 사망 관련 컬럼 리스트 (아오 많아)
death_log_cols = [
    'killer', 'killerCharacter', 'killerWeapon', 'causeOfDeath', 'placeOfDeath', 'killDetail',
    'killer2', 'killerCharacter2', 'killerWeapon2', 'causeOfDeath2', 'placeOfDeath2', 'killDetail2',
    'killer3', 'killerCharacter3', 'killerWeapon3', 'causeOfDeath3', 'placeOfDeath3', 'killDetail3'
]

# 결측치를 'None'으로 라벨링
for col in death_log_cols:
    if col in df.columns:
        df[col] = df[col].fillna('None')

In [5]:

# 조언 받아 'user1369612'으로 대체
df['nickname'] = df['nickname'].fillna('user1369612')

In [6]:
# language 필요할 것 같지 않으니 전체 열 삭제
df.drop(columns=['language'], inplace=True)

In [7]:
# 중복 데이터 제거
# 한 게임에 동일 유저가 두 명 들어가면 안 됨.
# userNum으로 해서 상관 없긴 한데, 이터널리턴은 다행히 중복닉 안 된다고 하네요
# gameId와 userNum의 조합 유니크하게
df = df.drop_duplicates(subset=['gameId', 'userNum'])

In [8]:
#게임이 랭크게임인지 판단
game_rank_flag = (
    df
    .groupby("gameId")
    .apply(
        lambda x: (
            (x["mmrGainInGame"] != 0).any() or
            (x["mmrLossEntryCost"] != 0).any()
        )
    )
    .reset_index(name="is_rank_game")
)

# 통째로 df1에 merge
df = df.merge(game_rank_flag, on="gameId", how="left")

# 랭크게임 여부를 확인하는 rank_type_v2 컬럼 추가
df["rank_type"] = df["is_rank_game"].map(
    {True: "rank_game", False: "rank_none"}
)

df["rank_type"].value_counts()

rank_type
rank_none    117995
rank_game     86430
Name: count, dtype: int64

In [9]:
#레벨 별로 나누어 랭크게임 가능/불가능 여부 판별

df["account_level_group"] = df["accountLevel"].apply(
    lambda x: "over 30" if x >= 30 else "under 30"
)


normal_game_df = df[df["rank_type"] == "rank_none"]

normal_game_count = (
    normal_game_df
    .groupby("account_level_group")
    ["gameId"]
    .nunique()
    .reset_index(name="normal_game_count")
)

normal_game_count

,account_level_group,normal_game_count
0,over 30,6167
1,under 30,5181


In [10]:
# 실력별 세그먼트 분류 위해 0점, 1~2000점, 
# 2001~4000점, 4001~6000점, 6001+로 5개의 세그먼트로 분류
# (이상치/결측치 없음, 분류 기준은 추후 변경 가능)
bins = [-1, 2400, 3600, 6400,np.inf]
labels = ['아이언&브론즈', '실버&골드', '플래티넘&다이아', '메테오라이트+']

df['rank_group'] = pd.cut(
    df['rankPoint'],
    bins=bins,
    labels=labels
    )


In [11]:
#캐릭별 status 매칭
import json

with open("character_stat.json", "r", encoding="utf-8") as f:
    j = json.load(f)

df_char = pd.DataFrame(j["data"])                 # code, , maxHp, attackPower, ...
df_char = df_char.rename(columns={"code": "characterNum", "name": "characterName_en"})

df["characterNum"] = pd.to_numeric(df["characterNum"], errors="coerce").astype("Int64")
df_char["characterNum"] = pd.to_numeric(df_char["characterNum"], errors="coerce").astype("Int64")

df = df.merge(df_char, on="characterNum", how="left")

In [12]:
#한글이름 매핑
charactor_n = {1: '재키',
2: '아야',
3:'피오라',
4:'매그너스',
5: '자히르',
6:'나딘',
7:'현우',
8:'하트',
9:'아이솔',
10:'리 다이린',
11:'유키',
12:'혜진',
13:'쇼우',
14:'키아라',
15:'시셀라',
 16:'실비아',
17:'아드리아나',
 18:'쇼이치',
 19:'엠마',
 20:'레녹스',
 21:'로지',
 22:'루크',
 23:'캐시',
 24:'아델라',
 25:'버니스',
 26:'바바라',
 27:'알렉스',
 28:'수아',
 29:'레온',
 30:'일레븐',
 31:'리오',
 32:'윌리엄',
 33:'니키',
 34:'나타폰',
 35:'얀',
 36:'이바',
 37:'다니엘',
 38:'제니',
 39:'카밀로',
 40:'클로에',
 41:'요한',
 42:'비앙카',
 43:'셀린',
 44:'에키온',
 45:'마이',
 46:'에이든',
 47:'라우라',
 48:'띠아',
 49:'펠릭스',
 50:'엘레나',
 51:'프리야',
 52:'아디나',
53:'마커스',
54:'칼라',
55:'에스텔',
56:'피올로',
57:'마르티나',
58:'헤이즈',
59:'아이작',
60:'타지아',
61:'이렘',
62:'테오도르',
63:'이안',
64:'바냐',
65:'데비&마를렌',
66:'아르다',
67:'아비게일',
68:'알론소',
69:'레니',
70:'츠바메',
71:'케네스',
72:'카티야',
73:'샬럿',
74:'다르코',
75:'르노어',
76:'가넷',
77:'유민',
78:'히스이',
79:'유스티나',
80:'이슈트반',
81:'니아',82:'슈린',83:'헨리',84:'블레어',85:'미르카',9999:'나쟈'}
df['characterName_kr'] = df['characterNum']
df['characterName_kr'] = df['characterName_kr'].map(charactor_n)

유저 대시보드 데이터 추출

In [13]:
# 유저 대시보드 데이터 추출
# 팀장님 감사합니다
# you're welcome

required_columns = [
    # 기본 식별 및 세그먼트
    'accountLevel', 'gameId', 'userNum', 'serverName', 'gameRank', 'rankPoint', 'matchingTeamMode', 
    'versionMajor', 'versionMinor',
    
    # 6대 KPI 계산용 (승률, KDA, DPM, 생존시간)
    'victory', 'playerKill', 'playerAssistant', 'playerDeaths', 
    'damageToPlayer', 'damageFromPlayer', 'playTime', 'survivableTime',
    
    # 팀 서포트 및 다양성 (회복, 보호막, 시야, 캐릭터)
    'teamRecover', 'protectAbsorb', 'viewContribution', 'characterNum',
    
    # 자원 전환 및 성장 효율
    'totalGainVFCredit', 'totalUseVFCredit',  # 총 획득/사용 크레딧
    'transferConsoleFromRevivalUseVFCredit', # 부활에 쓴 크레딧(안 필요할 수도 근데 그냥 제 경험 기반으로 넣어놓음)
    'transferConsoleFromMaterialUseVFCredit', # 재료에 쓴 크레딧
    'craftEpic', 'craftLegend', 'craftMythic', # 제작 아이템 등급별 수량
    
    # 개인 추세 및 MMR
    'mmrGainInGame', 'mmrLossEntryCost',
    
    #
    'rank_group','isLeavingBeforeCreditRevivalTerminate', 'giveUp', "account_level_group", "rank_type"

]

# 더 필요한 컬럼이 있다면 추후 추가하면 됨
# 태블로에서 데이터-새로고침 하면 됩니다

core_df = df[required_columns].copy()

core_df

,accountLevel,gameId,userNum,serverName,gameRank,rankPoint,matchingTeamMode,versionMajor,versionMinor,victory,playerKill,playerAssistant,playerDeaths,damageToPlayer,damageFromPlayer,playTime,survivableTime,teamRecover,protectAbsorb,viewContribution,characterNum,totalGainVFCredit,totalUseVFCredit,transferConsoleFromRevivalUseVFCredit,transferConsoleFromMaterialUseVFCredit,craftEpic,craftLegend,craftMythic,mmrGainInGame,mmrLossEntryCost,rank_group,isLeavingBeforeCreditRevivalTerminate,giveUp,account_level_group,rank_type
0,119,35260427,4378745,Seoul,5,5634,3,22,1,0,6,1,3,15206,14873,1125,23,0,3767,21,49,1209,1082,22,800,7,6,0,30,-44,플래티넘&다이아,False,0,over 30,rank_game
1,82,35260427,4313007,Seoul,4,5625,3,22,1,0,2,6,2,11130,16891,1220,23,0,2534,27,28,1265,998,168,600,6,5,0,69,-44,플래티넘&다이아,False,0,over 30,rank_game
2,133,35260427,4227659,Seoul,2,5285,3,22,1,0,2,8,3,16795,15347,1145,12,988,5744,18,23,1068,780,0,370,8,5,0,128,-43,플래티넘&다이아,False,0,over 30,rank_game
3,156,35260427,3979035,Seoul,1,5316,3,22,1,1,4,3,0,12282,15312,1342,60,0,3238,31,53,1302,1130,200,400,5,4,0,124,-43,플래티넘&다이아,False,0,over 30,rank_game
4,150,35260427,2118488,Seoul,1,5439,3,22,1,1,2,4,2,11339,15478,1342,60,0,0,25,65,1186,1120,0,650,7,6,0,124,-43,플래티넘&다이아,False,0,over 30,rank_game
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204420,10,35765219,3927798,Europe,3,0,3,23,0,0,0,0,1,0,0,838,0,0,0,0,68,464,0,0,0,0,0,0,0,0,아이언&브론즈,True,1,under 30,rank_none
204421,16,35783095,4932642,Asia,7,0,3,23,0,0,0,0,1,0,0,147,20,0,0,0,36,116,0,0,0,0,0,0,0,0,아이언&브론즈,True,1,under 30,rank_none
204422,118,35796912,456574,Asia,8,0,3,23,0,0,0,0,1,0,0,90,20,0,0,0,74,55,10,0,0,0,0,0,0,0,아이언&브론즈,True,1,over 30,rank_none
204423,56,35804415,4519640,Asia,8,4368,3,23,0,0,0,0,1,0,0,48,20,0,0,0,8,42,0,0,0,0,0,0,0,-72,플래티넘&다이아,True,1,over 30,rank_game


In [14]:
# 수치 통일 위해 정규화
core_df['scoreRecover'] = df['teamRecover'] / df['teamRecover'].replace(0, 1).max()
core_df['scoreProtect'] = df['protectAbsorb'] / df['protectAbsorb'].replace(0, 1).max()
core_df['scoreVision'] = df['viewContribution'] / df['viewContribution'].replace(0, 1).max()
core_df['scoreCreditRevive'] = df['creditRevivedOthersCount'] / df['creditRevivedOthersCount'].replace(0, 1).max()

In [15]:
#core_df 최종 저장 
core_df.to_csv('ER_core_data.csv', index=False, encoding='utf-8-sig')

콘텐츠 대시보드 데이터 추출

In [16]:
colunms_content = [
'characterName_kr',
'gameId'	,
'characterNum',	
'bestWeapon',
'gameRank',
'playerKill',	
'playerAssistant',	
'playerDeaths'	,
#'equipment'	,
'totalGainVFCredit'  ,
'killPlayerGainVFCredit',
'killChickenGainVFCredit',
'killBoarGainVFCredit',
'killWildDogGainVFCredit',
'killWolfGainVFCredit',
'killBearGainVFCredit',
'killOmegaGainVFCredit',
'killBatGainVFCredit',
'killWicklineGainVFCredit',
'killAlphaGainVFCredit',
'killItemBountyGainVFCredit',
'killDroneGainVFCredit',
'totalUseVFCredit'     ,
'remoteDroneUseVFCreditMySelf',
'remoteDroneUseVFCreditAlly',
'transferConsoleFromMaterialUseVFCredit',
'transferConsoleFromEscapeKeyUseVFCredit',
'transferConsoleFromRevivalUseVFCredit',
'tacticalSkillUpgradeUseVFCredit',
'viewContribution',
'maxHp_y',
'attackPower_y',
'defense_y',
'attackSpeed_y',
'attackRange',
'rank_type',
'versionMajor',
'versionMinor'
]

In [17]:
# ## df 저장

# df.shape
# df.to_csv("processed.csv", index=False)

In [18]:
#사용할 컬럼만
df_teab = df[colunms_content]

In [19]:
### ### ### ### ### ### ### ### ### ### ### ###
###               컨텐츠                     ###
###             전처리 코드                  ###
### ### ### ### ### ### ### ### ### ### ### ###

# 1) 랭크게임만
df_teab = df_teab[df_teab["rank_type"] == "rank_game"]

# 2) 게임 내 같은 팀원이 3명인 경우만 (gameId + gameRank 기준)
df_teab = (
    df_teab
    .groupby(["gameId", "gameRank"])
    .filter(lambda x: len(x) == 3)
)

# 3) 같은 팀원 추출
def extract_teammates(group):
    group = group.copy()
    for idx, row in group.iterrows():
        teammates = group.loc[group.index != idx, "characterName_kr"].tolist()
        teammates = sorted(teammates)
        group.loc[idx, "team_char_1"] = teammates[0]
        group.loc[idx, "team_char_2"] = teammates[1]
    return group

df_teab = (
    df_teab
    .groupby(["gameId", "gameRank"], group_keys=False)
    .apply(extract_teammates)
)


df_teab['teams_chars'] = df_teab['team_char_1']+df_teab['team_char_2']




In [20]:
df_teab.shape

(86142, 40)

In [21]:
# 수치 통일 위해 정규화
df_teab['scoreRecover'] = df['teamRecover'] / df['teamRecover'].replace(0, 1).max()
df_teab['scoreProtect'] = df['protectAbsorb'] / df['protectAbsorb'].replace(0, 1).max()
df_teab['scoreVision'] = df['viewContribution'] / df['viewContribution'].replace(0, 1).max()
df_teab['scoreCreditRevive'] = df['creditRevivedOthersCount'] / df['creditRevivedOthersCount'].replace(0, 1).max()

In [22]:
df_teab.to_csv('컨텐츠대시보드_데이터.csv', index=False,encoding='UTF-8')

In [23]:
### ### ### ### ### ### ### ### ### ### ### ###
###               컨텐츠                     ###
###             전처리 코드                  ###
### ### ### ### ### ### ### ### ### ### ### ###

In [24]:
import pandas as pd
import numpy as np
import re, ast, json

# =========================================================
# 0) 준비: df / df_teab 존재 전제
# =========================================================

CHAR_COL = "characterName_kr"
VICTORY_COL = "victory"
GAME_COL = "gameId"

# victory 정리
df[VICTORY_COL] = pd.to_numeric(df[VICTORY_COL], errors="coerce")

# =========================================================
# 1) 캐릭터별 주무기 픽률 / 승률 + 무기 한글명 -> df_teab에 추가
# =========================================================

# 무기 코드 -> 한글
weapon_type = {
    1:'글러브',2:'톤파',3:'방망이',4:'채찍',5:'투척',6:'암기',7:'활',8:'석궁',9:'권총',
    10:'돌격 소총',11:'저격총',13:'망치',14:'도끼',15:'단검',16:'양손검',17:'폴암',18:'쌍검',
    19:'창',20:'쌍절곤',21:'레이피어',22:'기타',23:'카메라',24:'아르카나'
}

# 재실행 안전: 기존 컬럼 제거
weapon_cols_to_drop = [
    "_bestWeapon_code", "bestWeapon_kr",
    "weapon_games", "character_total_games",
    "weapon_pick_rate_within_character", "weapon_win_rate"
]
df_teab.drop(columns=[c for c in weapon_cols_to_drop if c in df_teab.columns], inplace=True)

# 무기코드 통일(Int64)
df["_bestWeapon_code"] = pd.to_numeric(df["bestWeapon"], errors="coerce").astype("Int64")
df_teab["_bestWeapon_code"] = pd.to_numeric(df_teab["bestWeapon"], errors="coerce").astype("Int64")

# 무기 한글명
df_teab["bestWeapon_kr"] = df_teab["_bestWeapon_code"].map(weapon_type)

# 승률 계산은 victory가 0/1인 행만 사용
df_valid = df[df[VICTORY_COL].isin([0, 1])].copy()

WEAPON_KEY = "_bestWeapon_code"

weapon_stats = (
    df_valid.dropna(subset=[CHAR_COL, WEAPON_KEY])
            .groupby([CHAR_COL, WEAPON_KEY])
            .agg(
                weapon_games=(GAME_COL, "size"),
                weapon_wins=(VICTORY_COL, "sum"),
            )
            .reset_index()
)

char_tot = (
    df_valid.dropna(subset=[CHAR_COL])
            .groupby(CHAR_COL)
            .agg(character_total_games=(GAME_COL, "size"))
            .reset_index()
)

weapon_stats = weapon_stats.merge(char_tot, on=CHAR_COL, how="left")
weapon_stats["weapon_pick_rate_within_character"] = weapon_stats["weapon_games"] / weapon_stats["character_total_games"]
weapon_stats["weapon_win_rate"] = weapon_stats["weapon_wins"] / weapon_stats["weapon_games"]

# df_teab에 merge (캐릭터+무기코드)
df_teab = df_teab.merge(
    weapon_stats[[CHAR_COL, WEAPON_KEY,
                  "weapon_games", "character_total_games",
                  "weapon_pick_rate_within_character", "weapon_win_rate"]],
    on=[CHAR_COL, WEAPON_KEY],
    how="left"
)

# =========================================================
# 2) 아이템 코드 -> 한글명 딕셔너리 만들기 (raw_items)
# =========================================================

raw_items = """Item/Name/101101┃가위
;     Item/Name/101102┃만년필
;     Item/Name/101104┃식칼
;     Item/Name/101201┃군용 나이프
;     Item/Name/101202┃메스
;     Item/Name/101203┃자마다르
;     Item/Name/101301┃장미칼
;     Item/Name/101302┃스위스 아미 나이프
;     Item/Name/101303┃카라페이스 카타르
;     Item/Name/101401┃카른웬난
;     Item/Name/101402┃파산검
;     Item/Name/101404┃초진동나이프
;     Item/Name/101405┃프라가라흐
;     Item/Name/101406┃다마스커스 가시
;     Item/Name/101407┃마하라자
;     Item/Name/101408┃하이랜더 더크
;     Item/Name/101501┃월식
;     Item/Name/101502┃소울 리퍼
;     Item/Name/101503┃비색 단검
;     Item/Name/102101┃녹슨 검
;     Item/Name/102201┃장검
;     Item/Name/102301┃일본도
;     Item/Name/102401┃마사무네
;     Item/Name/102402┃무라마사
;     Item/Name/102403┃바스타드 소드
;     Item/Name/102404┃보검
;     Item/Name/102405┃뚜언 띠엔
;     Item/Name/102406┃아론다이트
;     Item/Name/102407┃엑스칼리버
;     Item/Name/102408┃플라즈마 소드
;     Item/Name/102409┃레바테인
;     Item/Name/102410┃모노호시자오
;     Item/Name/102411┃호푸어드
;     Item/Name/102412┃빛의 검
;     Item/Name/102413┃아케인 엣지
;     Item/Name/102501┃다인슬라이프
;     Item/Name/102502┃알마스
;     Item/Name/102503┃하데스 엣지
;     Item/Name/103201┃쌍칼
;     Item/Name/103202┃조잡한 쌍검
;     Item/Name/103301┃피렌체식 쌍검
;     Item/Name/103302┃쌍둥이 검
;     Item/Name/103401┃이천일류
;     Item/Name/103402┃자웅일대검
;     Item/Name/103403┃아수라
;     Item/Name/103404┃검은 나비
;     Item/Name/103501┃디오스쿠로이
;     Item/Name/103506┃훅소드
;     Item/Name/103507┃죽음의 발걸음
;     Item/Name/103502┃로이거 차르
;     Item/Name/103503┃간장과 막야
;     Item/Name/103504┃환영도
;     Item/Name/104101┃망치
;     Item/Name/104201┃워해머
;     Item/Name/104301┃모닝 스타
;     Item/Name/104302┃사슴 망치
;     Item/Name/104303┃운명의 망치
;     Item/Name/104401┃낭아봉
;     Item/Name/104402┃다그다의 망치
;     Item/Name/104403┃토르의 망치
;     Item/Name/104404┃개밥바라기
;     Item/Name/104405┃마법봉
;     Item/Name/104406┃천근추
;     Item/Name/104409┃아는 것이 힘
;     Item/Name/104501┃피스브링어
;     Item/Name/104502┃뿅망치
;     Item/Name/108503┃터럭손 방망이
;     Item/Name/108504┃로드 오브 하트
;     Item/Name/105102┃곡괭이
;     Item/Name/105103┃손도끼
;     Item/Name/105201┃사슬 낫
;     Item/Name/105202┃전투 도끼
;     Item/Name/105301┃경량화 도끼
;     Item/Name/105302┃사신의 낫
;     Item/Name/105401┃대부
;     Item/Name/105402┃빔 엑스
;     Item/Name/105403┃산타 무에르테
;     Item/Name/105404┃스퀴테
;     Item/Name/105405┃파라슈
;     Item/Name/105406┃하르페
;     Item/Name/105407┃저거너트
;     Item/Name/105408┃반고부
;     Item/Name/105501┃낙원의 낫
;     Item/Name/105502┃진홍빛 낫
;     Item/Name/107101┃단창
;     Item/Name/107201┃죽창
;     Item/Name/107301┃바이던트
;     Item/Name/107302┃파이크
;     Item/Name/107303┃도끼창
;     Item/Name/107401┃강창
;     Item/Name/107402┃애각창
;     Item/Name/107403┃장팔사모
;     Item/Name/107404┃코스믹 바이던트
;     Item/Name/107405┃트리아이나
;     Item/Name/107406┃화첨창
;     Item/Name/107407┃방천화극
;     Item/Name/107408┃청룡언월도
;     Item/Name/107409┃나기나타
;     Item/Name/107501┃롱기누스의 창
;     Item/Name/107502┃별 사냥꾼
;     Item/Name/108101┃나뭇가지
;     Item/Name/108102┃단봉
;     Item/Name/108103┃대나무
;     Item/Name/108104┃인체모형
;     Item/Name/108201┃먼지털이개
;     Item/Name/108202┃장봉
;     Item/Name/108301┃도깨비 방망이
;     Item/Name/108401┃우산
;     Item/Name/108402┃횃불
;     Item/Name/108405┃쇠파이프
;     Item/Name/108403┃구원의 여신상
;     Item/Name/108404┃타구봉
;     Item/Name/108501┃스파이의 우산
;     Item/Name/104407┃금강저
;     Item/Name/104408┃팔괘장
;     Item/Name/108502┃여의봉
;     Item/Name/109101┃채찍
;     Item/Name/109201┃오랏줄
;     Item/Name/109202┃철편
;     Item/Name/109301┃바람 채찍
;     Item/Name/109401┃뇌룡편
;     Item/Name/109402┃벽력편
;     Item/Name/109403┃글레이프니르
;     Item/Name/109404┃플라즈마 윕
;     Item/Name/109405┃캐소드라쉬
;     Item/Name/109406┃우라누스
;     Item/Name/109501┃혈화구절편
;     Item/Name/109502┃사복검
;     Item/Name/109503┃운명의 고리
;     Item/Name/110101┃너클
;     Item/Name/110102┃목장갑
;     Item/Name/110201┃글러브
;     Item/Name/110202┃아이언 너클
;     Item/Name/110301┃건틀릿
;     Item/Name/110302┃윙 너클
;     Item/Name/110401┃귀골 장갑
;     Item/Name/110402┃벽력귀투
;     Item/Name/110403┃유리 너클
;     Item/Name/110404┃회단 장갑
;     Item/Name/110405┃단영촌천투
;     Item/Name/110406┃디바인 피스트
;     Item/Name/110407┃블러드윙 너클
;     Item/Name/110408┃빙화현옥수
;     Item/Name/110409┃여래수투
;     Item/Name/110410┃브레이질 건틀릿
;     Item/Name/110411┃소수
;     Item/Name/110412┃천잠장갑
;     Item/Name/110501┃주작자문
;     Item/Name/110502┃프로스트팽
;     Item/Name/110503┃블러디 핸즈
;     Item/Name/111101┃맷손
;     Item/Name/111201┃톤파
;     Item/Name/111301┃경찰봉
;     Item/Name/111401┃류큐톤파
;     Item/Name/111402┃택티컬 톤파
;     Item/Name/111403┃마이쏙
;     Item/Name/111404┃플라즈마 톤파
;     Item/Name/111405┃윈드러너
;     Item/Name/111406┃홀스터 톤파
;     Item/Name/111501┃흑요석 짓테
;     Item/Name/111502┃만년한파
;     Item/Name/111503┃칼날 톤파
;     Item/Name/112101┃돌멩이
;     Item/Name/112103┃쇠구슬
;     Item/Name/112104┃유리병
;     Item/Name/401215┃달궈진 돌멩이
;     Item/Name/112105┃야구공
;     Item/Name/112202┃수류탄
;     Item/Name/112203┃화염병
;     Item/Name/112204┃슬링
;     Item/Name/112205┃싸인볼
;     Item/Name/112301┃밀가루 폭탄
;     Item/Name/112302┃소이탄
;     Item/Name/112303┃볼 라이트닝
;     Item/Name/112304┃플러버
;     Item/Name/112305┃고폭 수류탄
;     Item/Name/112306┃필럼
;     Item/Name/112401┃다비드슬링
;     Item/Name/112402┃연막탄
;     Item/Name/112403┃가시 탱탱볼
;     Item/Name/112404┃안티오크의 수류탄
;     Item/Name/112501┃루테늄 구슬
;     Item/Name/112405┃파이어 볼
;     Item/Name/112406┃프리즘 볼
;     Item/Name/112407┃아스트라페
;     Item/Name/112408┃점착폭탄
;     Item/Name/112502┃여의주
;     Item/Name/112503┃체이서
;     Item/Name/113101┃면도칼
;     Item/Name/113102┃트럼프 카드
;     Item/Name/113103┃CD
;     Item/Name/113104┃분필
;     Item/Name/113201┃다트
;     Item/Name/113202┃부적
;     Item/Name/113203┃빈티지 카드
;     Item/Name/113204┃토마호크
;     Item/Name/113205┃표창
;     Item/Name/113206┃흑건
;     Item/Name/113207┃매화비표
;     Item/Name/113301┃챠크람
;     Item/Name/113302┃유엽비도
;     Item/Name/113401┃미치광이왕의 카드
;     Item/Name/113402┃독침
;     Item/Name/113403┃법륜
;     Item/Name/113404┃플럼바타
;     Item/Name/113405┃옥전결
;     Item/Name/113406┃풍마 수리검
;     Item/Name/113407┃본크러셔
;     Item/Name/113408┃빙백은침
;     Item/Name/113409┃푸른색 단도
;     Item/Name/113410┃플레솃
;     Item/Name/113411┃건곤권
;     Item/Name/113412┃생사부
;     Item/Name/113501┃수다르사나
;     Item/Name/113502┃만천화우
;     Item/Name/113503┃흑련비표
;     Item/Name/114101┃양궁
;     Item/Name/114201┃목궁
;     Item/Name/114202┃장궁
;     Item/Name/114203┃컴포지트 보우
;     Item/Name/114301┃강궁
;     Item/Name/114302┃국궁
;     Item/Name/114303┃벽력궁
;     Item/Name/114304┃탄궁
;     Item/Name/114401┃편전
;     Item/Name/114402┃화전
;     Item/Name/114403┃골든래쇼 보우
;     Item/Name/114404┃큐피드의 활
;     Item/Name/114405┃트윈보우
;     Item/Name/114406┃제베의 활
;     Item/Name/114501┃엘리멘탈 보우
;     Item/Name/114502┃페일노트
;     Item/Name/114503┃아르기로톡소스
;     Item/Name/114504┃적궁백시
;     Item/Name/114407┃아르테미스
;     Item/Name/115101┃석궁
;     Item/Name/115201┃쇠뇌
;     Item/Name/115202┃크로스보우
;     Item/Name/115301┃노
;     Item/Name/115302┃저격궁
;     Item/Name/115303┃헤비 크로스보우
;     Item/Name/115401┃철궁
;     Item/Name/115402┃대황
;     Item/Name/115403┃발리스타
;     Item/Name/115404┃저격 크로스보우
;     Item/Name/115405┃영광금귀신기노
;     Item/Name/115406┃포이즌드 크로스보우
;     Item/Name/115501┃샤릉가
;     Item/Name/115502┃혈색반월
;     Item/Name/116101┃발터 PPK
;     Item/Name/116201┃매그넘-파이선
;     Item/Name/116202┃베레타 M92F
;     Item/Name/116301┃FN57
;     Item/Name/116401┃더블 리볼버 SP
;     Item/Name/116402┃매그넘-아나콘다
;     Item/Name/116408┃데린저
;     Item/Name/116403┃마탄의 사수
;     Item/Name/116404┃엘레강스
;     Item/Name/116405┃일렉트론 블라스터
;     Item/Name/116406┃매그넘-보아
;     Item/Name/116407┃글록 48
;     Item/Name/116409┃스탬피드
;     Item/Name/116410┃플라즈마 건
;     Item/Name/116501┃악켈테
;     Item/Name/116502┃알타이르
;     Item/Name/116503┃하이 눈
;     Item/Name/117101┃페도로프 자동소총
;     Item/Name/117201┃STG-44
;     Item/Name/117301┃AK-47
;     Item/Name/117401┃M16A1
;     Item/Name/117402┃개틀링 건
;     Item/Name/117403┃95식 자동 소총
;     Item/Name/117404┃AK-12
;     Item/Name/117405┃XCR
;     Item/Name/117406┃저지먼트
;     Item/Name/117407┃골드 러시
;     Item/Name/117501┃아그니
;     Item/Name/117502┃피안화
;     Item/Name/117503┃헬파이어
;     Item/Name/118101┃화승총
;     Item/Name/118201┃스프링필드
;     Item/Name/118301┃하푼건
;     Item/Name/118401┃금교전
;     Item/Name/118402┃레일건
;     Item/Name/118403┃Tac-50
;     Item/Name/118404┃인터벤션
;     Item/Name/118405┃NTW-20
;     Item/Name/118406┃폴라리스
;     Item/Name/118407┃가우스 라이플
;     Item/Name/118501┃사사성광
;     Item/Name/118502┃현자총통
;     Item/Name/118503┃안드로메다
;     Item/Name/118504┃위도우 메이커
;     Item/Name/119101┃쇠사슬
;     Item/Name/119201┃눈차크
;     Item/Name/119301┃샤퍼
;     Item/Name/119302┃블리더
;     Item/Name/119401┃대소반룡곤
;     Item/Name/119402┃초진동눈차크
;     Item/Name/119403┃케르베로스
;     Item/Name/119404┃블루3
;     Item/Name/119501┃히드라
;     Item/Name/119502┃비익련리
;     Item/Name/120101┃바늘
;     Item/Name/120201┃레이피어
;     Item/Name/120301┃블루밍
;     Item/Name/120302┃활빈검
;     Item/Name/120303┃에스톡
;     Item/Name/120401┃듀랜달 Mk2
;     Item/Name/120402┃미스틸테인
;     Item/Name/120403┃볼틱레토
;     Item/Name/120404┃유성검
;     Item/Name/120405┃주와이외즈
;     Item/Name/120406┃레드 팬서
;     Item/Name/120407┃에스프리
;     Item/Name/120408┃플랑베르주
;     Item/Name/120501┃아르고스의 눈
;     Item/Name/120502┃노스페라투
;     Item/Name/120503┃기어헬릭스
;     Item/Name/121101┃보급형 기타
;     Item/Name/121201┃골든 브릿지
;     Item/Name/121202┃싱글 픽업
;     Item/Name/121301┃루비 스페셜
;     Item/Name/121302┃험버커 픽업
;     Item/Name/121303┃King-V
;     Item/Name/121304┃노캐스터
;     Item/Name/121305┃슈퍼스트랫
;     Item/Name/121306┃야생마
;     Item/Name/121401┃보헤미안
;     Item/Name/121402┃천국의 계단
;     Item/Name/121403┃퍼플 헤이즈
;     Item/Name/121404┃새티스팩션
;     Item/Name/121405┃원더풀 투나잇
;     Item/Name/121406┃더 월
;     Item/Name/121407┃틴 스피릿
;     Item/Name/121501┃캡틴 페퍼
;     Item/Name/121502┃하트 브레이커
;     Item/Name/122101┃렌즈
;     Item/Name/122201┃카메라 건
;     Item/Name/122301┃컴팩트 카메라
;     Item/Name/122302┃레인지파인더
;     Item/Name/122303┃카메라 라이플
;     Item/Name/122401┃미러리스
;     Item/Name/122402┃컴파운드 사이트
;     Item/Name/122403┃카메라 캐논
;     Item/Name/122404┃V.I.C.G
;     Item/Name/122405┃폴라로이드 카메라
;     Item/Name/122501┃울트라비전
;     Item/Name/122502┃비전 플렉스
;     Item/Name/130101┃유리구슬
;     Item/Name/130201┃거울구슬
;     Item/Name/130202┃얼음구슬
;     Item/Name/130301┃의지의 지팡이
;     Item/Name/130302┃감정의 컵
;     Item/Name/130303┃이성의 칼
;     Item/Name/130304┃소유의 펜타클
;     Item/Name/130401┃은둔자
;     Item/Name/130402┃운명의 수레바퀴
;     Item/Name/130403┃절제
;     Item/Name/130404┃더 스타
;     Item/Name/130405┃더 문
;     Item/Name/130501┃여제
;     Item/Name/130502┃더 데스
;     Item/Name/131101┃카드모스의 부름 Lv2
;     Item/Name/131102┃카드모스의 부름 Lv3
;     Item/Name/131201┃바이퍼
;     Item/Name/131301┃데스애더
;     Item/Name/131302┃블랙맘바
;     Item/Name/131303┃사이드와인더
;     Item/Name/131401┃데스애더퀸
;     Item/Name/131402┃블랙맘바킹
;     Item/Name/131403┃슈퍼사이드와인더
;     Item/Name/131501┃데스애더퀸-MT
;     Item/Name/131502┃데스애더퀸-FC
;     Item/Name/131503┃데스애더퀸-VBS
;     Item/Name/131504┃블랙맘바킹-TL
;     Item/Name/131505┃블랙맘바킹-FC
;     Item/Name/131506┃블랙맘바킹-VBS
;     Item/Name/131507┃슈퍼사이드와인더-ML
;     Item/Name/131508┃슈퍼사이드와인더-FC
;     Item/Name/131509┃슈퍼사이드와인더-VBS
;     Item/Name/101701┃비색 단검-진홍
;     Item/Name/101702┃비색 단검-새벽
;     Item/Name/102701┃다인슬라이프-진홍
;     Item/Name/102702┃다인슬라이프-새벽
;     Item/Name/104701┃피스브링어-진홍
;     Item/Name/104702┃피스브링어-새벽
;     Item/Name/105701┃진홍빛 낫-진홍
;     Item/Name/105702┃진홍빛 낫-새벽
;     Item/Name/107701┃롱기누스의 창-진홍
;     Item/Name/107702┃롱기누스의 창-새벽
;     Item/Name/109701┃혈화구절편-진홍
;     Item/Name/109702┃혈화구절편-새벽
;     Item/Name/110701┃블러디 핸즈-진홍
;     Item/Name/110702┃블러디 핸즈-새벽
;     Item/Name/111701┃칼날 톤파-진홍
;     Item/Name/111702┃칼날 톤파-새벽
;     Item/Name/112701┃체이서-진홍
;     Item/Name/112702┃체이서-새벽
;     Item/Name/113701┃흑련비표-진홍
;     Item/Name/113702┃흑련비표-새벽
;     Item/Name/114701┃페일노트-진홍
;     Item/Name/114702┃페일노트-새벽
;     Item/Name/115701┃혈색반월-진홍
;     Item/Name/115702┃혈색반월-새벽
;     Item/Name/117701┃헬파이어-진홍
;     Item/Name/117702┃헬파이어-새벽
;     Item/Name/118701┃위도우 메이커-진홍
;     Item/Name/118702┃위도우 메이커-새벽
;     Item/Name/119701┃히드라-진홍
;     Item/Name/119702┃히드라-새벽
;     Item/Name/120701┃노스페라투-진홍
;     Item/Name/120702┃노스페라투-새벽
;     Item/Name/121701┃하트 브레이커-진홍
;     Item/Name/121702┃하트 브레이커-새벽
;     Item/Name/130701┃더 데스-진홍
;     Item/Name/130702┃더 데스-새벽
;     Item/Name/116701┃하이 눈-진홍
;     Item/Name/116702┃하이 눈-새벽
;     Item/Name/108701┃로드 오브 하트-진홍
;     Item/Name/108702┃로드 오브 하트-새벽
;     Item/Name/103701┃환영도-진홍
;     Item/Name/103702┃환영도-새벽
;     Item/Name/122701┃비전 플렉스-진홍
;     Item/Name/122702┃비전 플렉스-새벽
;     Item/Name/601501┃데스애더퀸-MT
;     Item/Name/601502┃데스애더퀸-FC
;     Item/Name/601503┃데스애더퀸-VBS
;     Item/Name/601504┃블랙맘바킹-TL
;     Item/Name/601505┃블랙맘바킹-FC
;     Item/Name/601506┃블랙맘바킹-VBS
;     Item/Name/601507┃슈퍼사이드와인더-ML
;     Item/Name/601508┃슈퍼사이드와인더-FC
;     Item/Name/601509┃슈퍼사이드와인더-VBS
;     Item/Name/201101┃머리띠
;     Item/Name/201102┃모자
;     Item/Name/201104┃자전거 헬멧
;     Item/Name/201111┃자연의 응답 Lv2
;     Item/Name/201112┃자연의 응답 Lv3
;     Item/Name/201201┃가면
;     Item/Name/201202┃머리테
;     Item/Name/201203┃베레모
;     Item/Name/201204┃사슬 코이프
;     Item/Name/201205┃안전모
;     Item/Name/201206┃피어나는 봉오리
;     Item/Name/201301┃방탄모
;     Item/Name/201302┃소방 헬멧
;     Item/Name/201303┃티아라
;     Item/Name/201304┃로빈
;     Item/Name/201305┃싱그러운 꽃잎
;     Item/Name/201401┃왕관
;     Item/Name/201402┃투구
;     Item/Name/201403┃미스릴 투구
;     Item/Name/201404┃수정 티아라
;     Item/Name/201405┃오토바이 헬멧
;     Item/Name/201406┃전술-OPS 헬멧
;     Item/Name/201407┃기사단장의 투구
;     Item/Name/201408┃월계관
;     Item/Name/201409┃제국 왕관
;     Item/Name/201410┃황실 부르고넷
;     Item/Name/201411┃변검
;     Item/Name/201412┃모호크 헬멧
;     Item/Name/201413┃비질란테
;     Item/Name/201414┃다이아뎀
;     Item/Name/201415┃성기사의 투구
;     Item/Name/201416┃만개하는 선율
;     Item/Name/201417┃카우보이 모자
;     Item/Name/201418┃플라즈마 투구
;     Item/Name/201501┃천사의 고리
;     Item/Name/201502┃빛의 증표
;     Item/Name/201503┃페르소나
;     Item/Name/201504┃예언자의 터번
;     Item/Name/201505┃레이싱 헬멧
;     Item/Name/201506┃황야의 별
;     Item/Name/201507┃우주 비행사의 헬멧
;     Item/Name/201508┃드워프의 투구
;     Item/Name/201509┃트와일라잇
;     Item/Name/201516┃천상의 메아리
;     Item/Name/201517┃천상의 메아리
;     Item/Name/202101┃바람막이
;     Item/Name/202103┃승복
;     Item/Name/202104┃백색 가운
;     Item/Name/202105┃전신 수영복
;     Item/Name/202106┃셔츠
;     Item/Name/202201┃가죽 갑옷
;     Item/Name/202202┃가죽 자켓
;     Item/Name/202203┃거북 도복
;     Item/Name/202205┃군복
;     Item/Name/202206┃덧댄 로브
;     Item/Name/202207┃드레스
;     Item/Name/202208┃드레스 셔츠
;     Item/Name/202209┃비키니
;     Item/Name/202210┃잠수복
;     Item/Name/202211┃사제복
;     Item/Name/202301┃라이더 자켓
;     Item/Name/202302┃사슬 갑옷
;     Item/Name/202303┃정장
;     Item/Name/202304┃치파오
;     Item/Name/202305┃판금 갑옷
;     Item/Name/202306┃한복
;     Item/Name/202307┃고위 사제복
;     Item/Name/202401┃방탄조끼
;     Item/Name/202402┃석양의 갑옷
;     Item/Name/202404┃어사의
;     Item/Name/202405┃광학미채 슈트
;     Item/Name/202406┃락커의 자켓
;     Item/Name/202407┃미스릴 갑옷
;     Item/Name/202408┃성기사의 갑옷
;     Item/Name/202409┃아름다운 갑옷
;     Item/Name/202410┃아마조네스 아머
;     Item/Name/202411┃용의 도복
;     Item/Name/202412┃지휘관의 갑옷
;     Item/Name/202413┃집사복
;     Item/Name/202415┃배틀 슈트
;     Item/Name/202416┃불꽃 드레스
;     Item/Name/202417┃EOD 슈트
;     Item/Name/202418┃턱시도
;     Item/Name/202419┃제사장의 예복
;     Item/Name/202420┃창파오
;     Item/Name/202421┃미스릴 크롭
;     Item/Name/202422┃방화복
;     Item/Name/202501┃카바나
;     Item/Name/202502┃퀸 오브 하트
;     Item/Name/202503┃성법의
;     Item/Name/202504┃버건디 47
;     Item/Name/202505┃아오자이
;     Item/Name/202506┃팬텀 자켓
;     Item/Name/202507┃가디언 슈트
;     Item/Name/202508┃진은 드레스
;     Item/Name/202509┃선녀강림
;     Item/Name/202510┃고스트
;     Item/Name/202511┃핏빛 망토
;     Item/Name/202512┃오메르타
;     Item/Name/202513┃슈팅스타 자켓
;     Item/Name/202514┃쿠튀리에
;     Item/Name/202515┃택티컬 아머
;     Item/Name/202516┃엘프 드레스
;     Item/Name/202517┃타이탄 아머
;     Item/Name/203101┃손목시계
;     Item/Name/203102┃붕대
;     Item/Name/203103┃토시
;     Item/Name/203104┃팔찌
;     Item/Name/203201┃가죽 방패
;     Item/Name/203202┃분대장 완장
;     Item/Name/203203┃브레이서
;     Item/Name/203204┃고장난 시계
;     Item/Name/203301┃검집
;     Item/Name/203302┃금팔찌
;     Item/Name/203303┃바주반드
;     Item/Name/203304┃진홍 팔찌
;     Item/Name/203305┃바브드 블로섬
;     Item/Name/203306┃포이즌드
;     Item/Name/203401┃강철 방패
;     Item/Name/203402┃소드 스토퍼
;     Item/Name/203403┃드라우프니르
;     Item/Name/203404┃미스릴 방패
;     Item/Name/203405┃바이탈 센서
;     Item/Name/203406┃기사의 신조
;     Item/Name/203407┃샤자한의 검집
;     Item/Name/203408┃큐브 워치
;     Item/Name/203409┃아이기스
;     Item/Name/203410┃틴달로스의 팔찌
;     Item/Name/203411┃나이팅게일
;     Item/Name/203412┃플라즈마 아크
;     Item/Name/203413┃텔루리안 타임피스
;     Item/Name/203414┃스마트 밴드
;     Item/Name/203415┃미닛맨의 표식
;     Item/Name/203501┃스카디의 팔찌
;     Item/Name/203502┃레이더
;     Item/Name/203503┃오토-암즈
;     Item/Name/203504┃프로미넌스
;     Item/Name/203505┃가시지네 견갑
;     Item/Name/203506┃스포츠 시계
;     Item/Name/203507┃틴달로스의 군주
;     Item/Name/203508┃샤를마뉴의 방패
;     Item/Name/203509┃혈사조
;     Item/Name/203510┃용의 비늘
;     Item/Name/203511┃서슬가시 체인
;     Item/Name/203512┃아흐라만의 손길
;     Item/Name/203513┃헬릭스
;     Item/Name/203514┃미스릴 완장
;     Item/Name/203515┃노바 실드
;     Item/Name/204101┃슬리퍼
;     Item/Name/204102┃운동화
;     Item/Name/204103┃타이즈
;     Item/Name/204201┃무릎 보호대
;     Item/Name/204202┃체인 레깅스
;     Item/Name/204203┃하이힐
;     Item/Name/204204┃힐리스
;     Item/Name/204205┃나막신
;     Item/Name/204301┃덧댄 슬리퍼
;     Item/Name/204302┃부츠
;     Item/Name/204303┃등산화
;     Item/Name/204304┃아이젠
;     Item/Name/204401┃강철 무릎 보호대
;     Item/Name/204402┃경량화 부츠
;     Item/Name/204403┃매버릭 러너
;     Item/Name/204404┃전투화
;     Item/Name/204405┃킬힐
;     Item/Name/204406┃풍화륜
;     Item/Name/204407┃미스릴 부츠
;     Item/Name/204408┃부케팔로스
;     Item/Name/204409┃EOD 부츠
;     Item/Name/204410┃글레이셜 슈즈
;     Item/Name/204411┃클링온 부츠
;     Item/Name/204412┃타키온 브레이스
;     Item/Name/204413┃탭루트
;     Item/Name/204414┃아이언 메이든
;     Item/Name/204415┃SCV
;     Item/Name/204416┃스텔라 스텝
;     Item/Name/204417┃카우보이 부츠
;     Item/Name/204501┃헤르메스의 부츠
;     Item/Name/204502┃분홍신
;     Item/Name/204503┃블레이드 부츠
;     Item/Name/204504┃알렉산드로스
;     Item/Name/204505┃칼날 다리
;     Item/Name/204506┃와일드 워커
;     Item/Name/204507┃갤럭시 스텝
;     Item/Name/204508┃미라지 워커
;     Item/Name/204509┃레이싱 부츠
;     Item/Name/204510┃엘디안 부츠
;     Item/Name/204511┃로즈 스텝
;     Item/Name/205101┃깃털
;     Item/Name/205102┃꽃
;     Item/Name/205103┃리본
;     Item/Name/205105┃부채
;     Item/Name/205106┃불경
;     Item/Name/205107┃상자
;     Item/Name/205108┃성배
;     Item/Name/205109┃십자가
;     Item/Name/205110┃쌍안경
;     Item/Name/205201┃백우선
;     Item/Name/205202┃성자의 유산
;     Item/Name/205203┃운명의 꽃
;     Item/Name/205204┃유리 조각
;     Item/Name/205205┃인형
;     Item/Name/205206┃저격 스코프
;     Item/Name/205207┃진신사리
;     Item/Name/205208┃화살통
;     Item/Name/205209┃먼지털이개
;     Item/Name/205210┃군선
;     Item/Name/205211┃비파단도
;     Item/Name/205212┃캐리비안 장식총
;     Item/Name/205213┃사격 교본
;     Item/Name/205301┃생명의 가루
;     Item/Name/205302┃우치와
;     Item/Name/205303┃탄창
;     Item/Name/205304┃궁기병의 화살통
;     Item/Name/205305┃월왕구천
;     Item/Name/205306┃해적의 증표
;     Item/Name/205307┃호크 아이
;     Item/Name/205308┃해적 깃발
;     Item/Name/205309┃오르골
;     Item/Name/205310┃능동 위장
;     Item/Name/205311┃마도서
;     Item/Name/205312┃아이테르 깃털
;     Item/Name/205313┃파일 벙커
;     Item/Name/205401┃달빛 펜던트
;     Item/Name/205402┃만년빙
;     Item/Name/205403┃삼매진화
;     Item/Name/205404┃슈뢰딩거의 상자
;     Item/Name/205405┃진리는 나의 빛
;     Item/Name/205406┃요명월
;     Item/Name/205407┃미스릴 퀴버
;     Item/Name/205408┃살라딘의 화살통
;     Item/Name/205409┃살라딘의 화살통 Mk2
;     Item/Name/205501┃운명의 주사위
;     Item/Name/205502┃파초선
;     Item/Name/205503┃쿤달라
;     Item/Name/205504┃오르비스
;     Item/Name/205505┃호루스의 눈
;     Item/Name/205506┃쿤달라 Mk2
;     Item/Name/205507┃네크로노미콘
;     Item/Name/205508┃에메랄드 타블렛
;     Item/Name/302101┃꿀
;     Item/Name/302102┃물
;     Item/Name/302103┃얼음
;     Item/Name/302104┃위스키
;     Item/Name/302106┃커피콩
;     Item/Name/302107┃탄산수
;     Item/Name/302108┃우유
;     Item/Name/302201┃뜨거운 물
;     Item/Name/302202┃레몬에이드
;     Item/Name/302203┃물병
;     Item/Name/302204┃백주
;     Item/Name/302205┃소주
;     Item/Name/302206┃아이스 커피
;     Item/Name/302207┃칵테일
;     Item/Name/302208┃커피 리큐르
;     Item/Name/302209┃콜라
;     Item/Name/302210┃카페라테
;     Item/Name/302211┃꿀탄 우유
;     Item/Name/302213┃하이볼
;     Item/Name/302214┃초코 우유
;     Item/Name/302215┃꿀물
;     Item/Name/302216┃얼음물
;     Item/Name/302217┃온더락
;     Item/Name/302218┃카우보이
;     Item/Name/302219┃제로 탄산수
;     Item/Name/302220┃티어드롭
;     Item/Name/302221┃꿀스키
;     Item/Name/302222┃슬러시
;     Item/Name/302223┃사이다
;     Item/Name/302301┃고량주
;     Item/Name/302302┃뜨거운 꿀물
;     Item/Name/302303┃백일취
;     Item/Name/302304┃아메리카노
;     Item/Name/302305┃약주
;     Item/Name/302307┃위스키 콕
;     Item/Name/302308┃정화수
;     Item/Name/302309┃캔 콜라
;     Item/Name/302310┃핫초코
;     Item/Name/302311┃깔루아 밀크
;     Item/Name/302312┃셀레네의 눈물
;     Item/Name/302313┃시원한 콜라
;     Item/Name/302314┃위스키 핫토디
;     Item/Name/302315┃스파클 히트
;     Item/Name/302316┃스팀 밀크
;     Item/Name/301102┃감자
;     Item/Name/301104┃대구
;     Item/Name/301105┃레몬
;     Item/Name/301106┃마늘
;     Item/Name/301107┃반창고
;     Item/Name/301109┃붕어
;     Item/Name/301110┃빵
;     Item/Name/301111┃고기
;     Item/Name/301112┃달걀
;     Item/Name/301113┃생라면
;     Item/Name/301116┃약초
;     Item/Name/301119┃초콜릿
;     Item/Name/301120┃카레 가루
;     Item/Name/301121┃양송이 버섯
;     Item/Name/301201┃꿀 바른 대구살
;     Item/Name/301204┃대구 간 통조림
;     Item/Name/301205┃마늘빵
;     Item/Name/301206┃버터
;     Item/Name/301207┃보약
;     Item/Name/301209┃붕어빵
;     Item/Name/301305┃성수
;     Item/Name/301213┃지혈제
;     Item/Name/301216┃초코파이
;     Item/Name/301217┃한방침
;     Item/Name/301218┃난초
;     Item/Name/301222┃탄두리
;     Item/Name/301224┃마늘 베이컨 말이
;     Item/Name/301225┃번
;     Item/Name/301226┃햄버거
;     Item/Name/301227┃감자빵
;     Item/Name/301228┃감자스프
;     Item/Name/301229┃달걀 생선 필레
;     Item/Name/301230┃시트러스 케이크
;     Item/Name/301231┃레몬 커스터드
;     Item/Name/301232┃마늘 꿀절임
;     Item/Name/301233┃꿀바른 붕어
;     Item/Name/301234┃계란빵
;     Item/Name/301235┃<color=yellow>이스터 에그</color>
;     Item/Name/301236┃위스키 봉봉
;     Item/Name/301237┃초코 아이스크림
;     Item/Name/301238┃카레빵
;     Item/Name/301239┃눈물젖은 빵
;     Item/Name/301240┃연어 샌드위치
;     Item/Name/301241┃야키소바 빵
;     Item/Name/301242┃포카치아
;     Item/Name/301243┃레어 스테이크
;     Item/Name/301244┃메로구이
;     Item/Name/301245┃덜 구운 감자
;     Item/Name/301246┃사과사탕
;     Item/Name/301301┃매운탕
;     Item/Name/301302┃감자튀김
;     Item/Name/301303┃구운 감자
;     Item/Name/301304┃구운 붕어
;     Item/Name/301306┃메로구이
;     Item/Name/301307┃뜨거운 라면
;     Item/Name/301308┃모카빵
;     Item/Name/301309┃스크램블 에그
;     Item/Name/301311┃초코칩 쿠키
;     Item/Name/301312┃초코파이 상자
;     Item/Name/301313┃카레
;     Item/Name/301314┃탕약
;     Item/Name/301315┃허니버터
;     Item/Name/301316┃후라이드 치킨
;     Item/Name/301317┃힐링 포션
;     Item/Name/301318┃삶은 달걀
;     Item/Name/301319┃술빵
;     Item/Name/301322┃카레 고로케
;     Item/Name/301323┃웰던 스테이크
;     Item/Name/301324┃구급상자
;     Item/Name/301325┃버터 감자구이
;     Item/Name/301326┃생선까스
;     Item/Name/301327┃볶음 라면
;     Item/Name/301328┃냉면
;     Item/Name/301329┃대환단
;     Item/Name/301330┃생일 케이크
;     Item/Name/301337┃따끈따끈한 토스트
;     Item/Name/301338┃루텔라
;     Item/Name/301339┃구운 마늘
;     Item/Name/301340┃베이컨 토스트
;     Item/Name/301346┃웰던 스테이크
;     Item/Name/301348┃1주년 케이크
;     Item/Name/301349┃양송이 꼬치 구이
;     Item/Name/301401┃피쉬 앤 칩스
;     Item/Name/301331┃마늘라면
;     Item/Name/301332┃뻥이요
;     Item/Name/301333┃눈꽃
;     Item/Name/301403┃일레븐 세트
;     Item/Name/301405┃연어
;     Item/Name/301406┃호박 고구마
;     Item/Name/301407┃냉동 피자
;     Item/Name/301408┃민트 초코
;     Item/Name/301409┃트러플
;     Item/Name/301501┃연어 스테이크
;     Item/Name/301502┃하와이안 피자
;     Item/Name/301503┃꿀고구마
;     Item/Name/301504┃트러플 파스타
;     Item/Name/301601┃백년 수프
;     Item/Name/301602┃만년 수프
;     Item/Name/802601┃컴뱃 에피네프린
;     Item/Name/303501┃현상금
;     Item/Name/303502┃크레딧+
;     Item/Name/303503┃크레딧++
;     Item/Name/303504┃크레딧
;     Item/Name/303505┃크레딧
;     Item/Name/303506┃크레딧
;     Item/Name/303507┃크레딧
;     Item/Name/303508┃크레딧
;     Item/Name/303509┃크레딧
;     Item/Name/306001┃가젯 에너지 팩
;     Item/Name/401101┃못
;     Item/Name/401103┃가죽
;     Item/Name/401104┃거북이 등딱지
;     Item/Name/401105┃고무
;     Item/Name/401106┃고철
;     Item/Name/401107┃라이터
;     Item/Name/401108┃레이저 포인터
;     Item/Name/401109┃마패
;     Item/Name/401110┃배터리
;     Item/Name/401111┃알코올
;     Item/Name/401112┃오일
;     Item/Name/401113┃옷감
;     Item/Name/401114┃원석
;     Item/Name/401116┃접착제
;     Item/Name/401117┃종이
;     Item/Name/401118┃철광석
;     Item/Name/401120┃캔
;     Item/Name/401121┃화약
;     Item/Name/401122┃고장난 노트북
;     Item/Name/401123┃화학품
;     Item/Name/401124┃흑연
;     Item/Name/401219┃다이아몬드
;     Item/Name/401201┃강철
;     Item/Name/401202┃기름먹인 천
;     Item/Name/401203┃뜨거운 오일
;     Item/Name/401217┃루비
;     Item/Name/401205┃방전 전지
;     Item/Name/401206┃분필 가루
;     Item/Name/401208┃생명의 나무
;     Item/Name/401209┃운석
;     Item/Name/401210┃재
;     Item/Name/401211┃전자 부품
;     Item/Name/401212┃정교한 도면
;     Item/Name/401213┃철판
;     Item/Name/401214┃황금
;     Item/Name/401216┃철사
;     Item/Name/401218┃하드커버
;     Item/Name/401301┃문스톤
;     Item/Name/401302┃독약
;     Item/Name/401303┃모터
;     Item/Name/401304┃미스릴
;     Item/Name/401305┃유리판
;     Item/Name/401306┃이온 전지
;     Item/Name/401307┃덧댄 가죽
;     Item/Name/401308┃두꺼운가죽
;     Item/Name/401401┃VF 혈액 샘플
;     Item/Name/401402┃진화의 돌
;     Item/Name/401403┃포스 코어
;     Item/Name/401405┃진홍의 샤드
;     Item/Name/401406┃새벽빛 샤드
;     Item/Name/401407┃균열석
;     Item/Name/409401┃재생 팔찌
;     Item/Name/409402┃시스템암호코드
;     Item/Name/409403┃루트킷
;     Item/Name/501101┃{0}의 재생 팔찌
;     Item/Name/501201┃방전된 노트북
;     Item/Name/501401┃휴대폰
;     Item/Name/501501┃노트북
;     Item/Name/501402┃시스템 암호 코드
;     Item/Name/501502┃네트워크 PC
;     Item/Name/502101┃감시 카메라
;     Item/Name/502102┃올가미
;     Item/Name/502103┃쥐덫
;     Item/Name/502104┃피아노선
;     Item/Name/502105┃간이 모닥불 키트
;     Item/Name/502201┃가시 발판
;     Item/Name/502202┃개량형 쥐덫
;     Item/Name/502203┃다이너마이트
;     Item/Name/502204┃대나무 트랩
;     Item/Name/502205┃부비트랩
;     Item/Name/502206┃소란 발생기
;     Item/Name/502207┃망원 카메라
;     Item/Name/502208┃정찰 드론
;     Item/Name/502209┃위장 카메라
;     Item/Name/502210┃모닥불
;     Item/Name/502211┃고배율 카메라
;     Item/Name/502212┃망원 카메라
;     Item/Name/502301┃가시구슬
;     Item/Name/502302┃메이지 룰렛
;     Item/Name/502303┃정글 기요틴
;     Item/Name/502304┃지뢰
;     Item/Name/502305┃펜듈럼 도끼
;     Item/Name/502306┃폭발 트랩
;     Item/Name/502307┃RDX
;     Item/Name/502308┃EMP 드론
;     Item/Name/502310┃지압판 트랩
;     Item/Name/502402┃폭뢰침
;     Item/Name/502403┃화염 트랩
;     Item/Name/502404┃C-4
;     Item/Name/502405┃더블 기요틴
;     Item/Name/502406┃크레모어
;     Item/Name/502407┃히든 메이든
;     Item/Name/502501┃리모트 마인
;     Item/Name/502502┃스마트 폭탄
;     Item/Name/503101┃키오스크 호출기
;     Item/Name/503102┃플레어건 B
;     Item/Name/503103┃플레어건 R
;     Item/Name/503104┃휴대용 VLS
;     Item/Name/503105┃루프 카드키
;     Item/Name/503106┃루프 카드키 - γ
;     Item/Name/503107┃루프 카드키 - RED
;     Item/Name/503108┃사냥꾼의 솥단지
;     Item/Name/504101┃아글라이아의 선물 - 일반
;     Item/Name/504201┃아글라이아의 선물 - 고급
;     Item/Name/504301┃아글라이아의 선물 - 희귀
;     Item/Name/504401┃톤파 보급 상자
;     Item/Name/504402┃양손검 보급 상자
;     Item/Name/504403┃암기 보급 상자
;     Item/Name/504404┃권총 보급 상자
;     Item/Name/504405┃아글라이아의 선물 - 영웅
;     Item/Name/504501┃아글라이아의 선물 - 전설
;     Item/Name/504602┃휴대용 생성기 - 머리
;     Item/Name/504601┃휴대용 생성기 - 옷
;     Item/Name/504603┃휴대용 생성기 - 팔
;     Item/Name/504604┃휴대용 생성기 - 다리
;     Item/Name/504605┃아글라이아의 선물 - 초월
;     Item/Name/540001┃오뜨꾸뛰르
;     Item/Name/999999┃전술 강화 모듈
;     Item/Name/201518┃쿼드 아이
;     Item/Name/201519┃검은 죽음
;     Item/Name/201520┃사이버 스토커
;     Item/Name/202518┃레버넌트
;     Item/Name/201522┃커맨더 헤드셋
;     Item/Name/202521┃레이싱 슈트
;     Item/Name/202522┃유령 신부의 드레스
;     Item/Name/705607┃토템
;     Item/Name/705608┃임세티
;     Item/Name/705620┃천룡잠
;     Item/Name/205601┃요술램프
;     Item/Name/201521┃블래스터 헬멧
;     Item/Name/202526┃이단심판관
;     Item/Name/201419┃용접 마스크
;     Item/Name/201523┃스파크 실드
;     Item/Name/202523┃길리 슈트
;     Item/Name/121503┃헤븐즈 도어
;     Item/Name/108505┃시체매
;     Item/Name/105503┃네플리온
;     Item/Name/201524┃바니햇
;     Item/Name/202524┃스펙터
;     Item/Name/202525┃현자의 로브
;     Item/Name/202527┃흑염룡의 갑주
;     Item/Name/201701┃핏빛 왕관
;     Item/Name/121504┃프리 버드
;     Item/Name/114505┃이글 아이
;     Item/Name/110504┃아이언 피스트
;     Item/Name/204418┃글레디에이터
;     Item/Name/204419┃델타 레드
;     Item/Name/201525┃레가투스
;     Item/Name/104503┃둠스데이
;     Item/Name/118505┃데드 아이
;     Item/Name/117504┃엔데버 AX
;     Item/Name/101504┃악의
;     Item/Name/130503┃더 썬
;     Item/Name/111504┃맨티스 톤파
;     Item/Name/104504┃레비아탄
;     Item/Name/117505┃발키리
;     Item/Name/103505┃타나토스
;     Item/Name/119503┃게리온
;     Item/Name/119504┃네온 체인
;     Item/Name/115503┃템페스트
;     Item/Name/115504┃팬텀 볼트
;     Item/Name/122503┃온 에어
;     Item/Name/122504┃트리플 포커스
;     Item/Name/201526┃앙 가르드
;     Item/Name/201527┃인사이트
;     Item/Name/201528┃발할라의 투구
;     Item/Name/201529┃아이실드
;     Item/Name/201420┃마녀 모자
;     Item/Name/201421┃여우 가면
;     Item/Name/201530┃다비드 헤드폰
;     Item/Name/201422┃스포츠 선글라스
;     Item/Name/201423┃전술 고글
;     Item/Name/201531┃마린 햇
;     Item/Name/201532┃캡틴 햇
;     Item/Name/201533┃검은 베일
;     Item/Name/204512┃스페이스 부츠
;     Item/Name/201534┃붉은 별빛
;     Item/Name/202528┃페더폴
;     Item/Name/202529┃화령장
;     Item/Name/203516┃파프니르
;     Item/Name/204513┃픽시 부츠
;     Item/Name/202415_D┃배틀 슈트 [다비드]
;     Item/Name/202407_D┃미스릴 갑옷 [다비드]
;     Item/Name/202416_D┃불꽃 드레스 [다비드]
;     Item/Name/202421_D┃미스릴 크롭 [다비드]
;     Item/Name/202501_D┃카바나 [다비드]
;     Item/Name/202503_D┃성법의 [다비드]
;     Item/Name/202505_D┃아오자이 [다비드]
;     Item/Name/202506_D┃팬텀 자켓 [다비드]
;     Item/Name/202507_D┃가디언 슈트 [다비드]
;     Item/Name/202508_D┃진은 드레스 [다비드]
;     Item/Name/202509_D┃선녀강림 [다비드]
;     Item/Name/202510_D┃고스트 [다비드]
;     Item/Name/202511_D┃핏빛 망토 [다비드]
;     Item/Name/202512_D┃오메르타 [다비드]
;     Item/Name/202513_D┃슈팅스타 자켓 [다비드]
;     Item/Name/202514_D┃쿠튀리에 [다비드]
;     Item/Name/202515_D┃택티컬 아머 [다비드]
;     Item/Name/202516_D┃엘프 드레스 [다비드]
;     Item/Name/202517_D┃타이탄 아머 [다비드]
;     Item/Name/202523_D┃길리 슈트 [다비드]
;     Item/Name/202524_D┃스펙터 [다비드]
;     Item/Name/202525_D┃현자의 로브 [다비드]
;     Item/Name/202209_D┃비키니 [다비드]
;     Item/Name/202504_D┃버건디 47 [다비드]
;     Item/Name/202502_D┃퀸 오브 하트 [다비드]
;     Item/Name/202521_D┃레이싱 슈트 [다비드]
;     Item/Name/202522_D┃유령 신부의 드레스 [다비드]
;     Item/Name/202518_D┃레버넌트 [다비드]
;     Item/Name/202526_D┃이단심판관 [다비드]
;     Item/Name/202527_D┃흑염룡의 갑주 [다비드]
;     Item/Name/201534_D┃붉은 별빛 [다비드]
;     Item/Name/202529_D┃화령장 [다비드]
;     Item/Name/602409┃레바테인 Mk2
;     Item/Name/607406┃화첨창 Mk2
;     Item/Name/610501┃주작자문 Mk2
;     Item/Name/701451┃택티컬 바이저
;     Item/Name/702503┃성법의 Mk2
;     Item/Name/705504┃오르비스
;     Item/Name/705502┃파초선 Mk2
;     Item/Name/705604┃클라다 반지
;     Item/Name/705603┃하트 온 파이어
;     Item/Name/705605┃머큐리 
;     Item/Name/705606┃아크 리액터
;     Item/Name/705609┃프시케의 칼날 
;     Item/Name/705610┃영웅의 기록 
;     Item/Name/705611┃E.M.O.T.E
;     Item/Name/705612┃클라다 반지 Mk2
;     Item/Name/705613┃아크 리액터 Mk2
;     Item/Name/705614┃프시케의 칼날
;     Item/Name/705615┃영웅의 기록 Mk2
;     Item/Name/705616┃E.M.O.T.E Mk2
;     Item/Name/705618┃요술 램프
;     Item/Name/705619┃별 조각
;     Infusion/Item/705504┃아티팩트 Mk2
;     Infusion/Item/705601┃미니어쳐 솔라 시스템
;     Infusion/Item/705602┃코발트블루
;     Infusion/Item/705502┃파초선 Mk2
;     Infusion/Item/705607┃토템 
;     Infusion/Item/705604┃클라다 반지
;     Infusion/Item/705603┃하트 온 파이어
;     Infusion/Item/705605┃머큐리
;     Infusion/Item/705606┃아크 리액터
;     Infusion/Item/705608┃임세티
;     Infusion/Item/705610┃영웅의 기록
;     Infusion/Item/705611┃E.M.O.T.E
;     Infusion/Item/705612┃클라다 반지 Mk2
;     Infusion/Item/705613┃아크 리액터 Mk2
;     Infusion/Item/705614┃프시케의 칼날 Mk2
;     Infusion/Item/705615┃영웅의 기록 Mk2
;     Infusion/Item/705616┃E.M.O.T.E Mk2
;     Item/Name/202601┃배틀 슈트 [다비드]
;     Item/Name/202602┃미스릴 갑옷 [다비드]
;     Item/Name/202603┃불꽃 드레스 [다비드]
;     Item/Name/202604┃미스릴 크롭 [다비드]
;     Item/Name/202605┃카바나 [다비드]
;     Item/Name/202606┃성법의 [다비드]
;     Item/Name/202607┃아오자이 [다비드]
;     Item/Name/202608┃팬텀 자켓 [다비드]
;     Item/Name/202609┃가디언 슈트 [다비드]
;     Item/Name/202610┃진은 드레스 [다비드]
;     Item/Name/202611┃선녀강림 [다비드]
;     Item/Name/202612┃고스트 [다비드]
;     Item/Name/202613┃핏빛 망토 [다비드]
;     Item/Name/202614┃오메르타 [다비드]
;     Item/Name/202615┃슈팅스타 자켓 [다비드]
;     Item/Name/202616┃쿠튀리에 [다비드]
;     Item/Name/202617┃택티컬 아머 [다비드]
;     Item/Name/202618┃엘프 드레스 [다비드]
;     Item/Name/202619┃타이탄 아머 [다비드]
;     Item/Name/202620┃길리 슈트 [다비드]
;     Item/Name/202621┃스펙터 [다비드]
;     Item/Name/202622┃현자의 로브 [다비드]
;     Item/Name/202623┃비키니 [다비드]
;     Item/Name/202624┃버건디 47 [다비드]
;     Item/Name/202625┃퀸 오브 하트 [다비드]
;     Item/Name/202626┃레이싱 슈트 [다비드]
;     Item/Name/202627┃유령 신부의 드레스 [다비드]
;     Item/Name/202628┃레버넌트 [다비드]
;     Item/Name/202629┃이단심판관 [다비드]
;     Item/Name/705601┃미니어쳐 솔라 시스템
;     Item/Name/705602┃코발트블루
;     Item/Name/702601┃이단심판관
"""

item_name_map = {}
for line in raw_items.splitlines():
    line = line.strip()
    if not line or "┃" not in line:
        continue
    left, name = line.split("┃", 1)
    code = left.split("/")[-1].strip()     # "101101" 또는 "202415_D"
    item_name_map[code] = name.strip()

# =========================================================
# 3) 캐릭터별 장비조합 승률 Top3 -> df_teab에 (Top1~Top3 컬럼 9개) 추가
# =========================================================

# 재실행 안전: 기존 Top3 컬럼 제거
prefixes = ("equip_combo_kr_top", "win_rate_top", "games_top")
drop_cols = [c for c in df_teab.columns if c.startswith(prefixes)]
df_teab.drop(columns=drop_cols, inplace=True, errors="ignore")

# --- equipment 파서: 어떤 형태든 리스트로 뽑아내기 ---
def parse_equipment(x):
    if pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(i).strip() for i in x if str(i).strip()]

    s = str(x).strip()
    if not s or s.lower() in ["none", "nan"]:
        return []

    # 이미 "112302 | 201410 | ..." 형태면 바로 split
    if " | " in s:
        return [p.strip() for p in s.split(" | ") if p.strip()]

    # JSON / literal
    for loader in (lambda t: json.loads(t), lambda t: ast.literal_eval(t)):
        try:
            obj = loader(s)
            if isinstance(obj, list):
                return [str(i).strip() for i in obj if str(i).strip()]
            if isinstance(obj, dict):
                vals = list(obj.values())
                flat = []
                for v in vals:
                    if isinstance(v, (list, tuple)):
                        flat.extend(v)
                    else:
                        flat.append(v)
                return [str(i).strip() for i in flat if str(i).strip()]
        except Exception:
            pass

    # 구분자 분해
    parts = re.split(r"[,\|\+/;]", s)
    return [p.strip() for p in parts if p.strip()]

def normalize_item_code(token):
    s = str(token).strip()
    if not s:
        return ""
    # "Item/Name/101101" 같은 형태면 마지막만
    if "/" in s:
        s = s.split("/")[-1]
    return s.strip()

SEP = " | "

def combo_code_to_name(combo, mapping):
    if pd.isna(combo):
        return pd.NA
    codes = [c.strip() for c in str(combo).split(SEP)]
    names = [mapping.get(c, c) for c in codes]
    return SEP.join(names)

# --- 조합 만들기(코드 문자열 " | "로 통일) ---
tmp = df[[CHAR_COL, VICTORY_COL, "equipment"]].copy()
tmp = tmp[tmp[VICTORY_COL].isin([0, 1])].copy()

tmp["_equip_list"] = tmp["equipment"].apply(parse_equipment)
tmp["_equip_codes"] = tmp["_equip_list"].apply(lambda lst: [normalize_item_code(x) for x in lst])
tmp["_equip_codes"] = tmp["_equip_codes"].apply(lambda lst: sorted([x for x in lst if x]))  # 순서 무시(정렬)

tmp["equip_combo"] = tmp["_equip_codes"].apply(lambda lst: SEP.join(lst))
tmp = tmp[tmp["equip_combo"].str.len() > 0].copy()

# --- 캐릭터-조합별 games / win_rate ---
combo_stats = (
    tmp.groupby([CHAR_COL, "equip_combo"])
       .agg(games=(VICTORY_COL, "size"),
            win_rate=(VICTORY_COL, "mean"))
       .reset_index()
)

# 표본 너무 작은 조합 제외
MIN_GAMES = 20
combo_stats = combo_stats[combo_stats["games"] >= MIN_GAMES].copy()

# 승률 내림차순, 동률이면 games 많은 조합 우선
combo_stats = combo_stats.sort_values([CHAR_COL, "win_rate", "games"], ascending=[True, False, False])
combo_stats["top3"] = combo_stats.groupby(CHAR_COL).cumcount() + 1

# Top3만 남기기
top3_df = combo_stats[combo_stats["top3"].isin([1, 2, 3])].copy()

# 조합 한글명 컬럼
top3_df["equip_combo_kr"] = top3_df["equip_combo"].apply(lambda x: combo_code_to_name(x, item_name_map))

# --- wide로 변환(Top1~Top3만 컬럼 9개 생성) ---
wide = top3_df.pivot_table(
    index=CHAR_COL,
    columns="top3",
    values=["equip_combo_kr", "win_rate", "games"],
    aggfunc="first"
)
wide.columns = [f"{v}_top{int(k)}" for v, k in wide.columns]
wide = wide.reset_index()

# df_teab에 캐릭터 기준 merge (모든 행에 반복 저장됨)
df_teab = df_teab.merge(wide, on=CHAR_COL, how="left")

# =========================================================
# 4) 최종 저장 
# =========================================================
df_teab.to_csv("컨텐츠대시보드_데이터.csv", index=False, encoding="UTF-8")
print(df_teab.shape)



(86142, 59)


In [25]:
df_teab

,characterName_kr,gameId,characterNum,bestWeapon,gameRank,playerKill,playerAssistant,playerDeaths,totalGainVFCredit,killPlayerGainVFCredit,killChickenGainVFCredit,killBoarGainVFCredit,killWildDogGainVFCredit,killWolfGainVFCredit,killBearGainVFCredit,killOmegaGainVFCredit,killBatGainVFCredit,killWicklineGainVFCredit,killAlphaGainVFCredit,killItemBountyGainVFCredit,killDroneGainVFCredit,totalUseVFCredit,remoteDroneUseVFCreditMySelf,remoteDroneUseVFCreditAlly,transferConsoleFromMaterialUseVFCredit,transferConsoleFromEscapeKeyUseVFCredit,transferConsoleFromRevivalUseVFCredit,tacticalSkillUpgradeUseVFCredit,viewContribution,maxHp_y,attackPower_y,defense_y,attackSpeed_y,attackRange,rank_type,versionMajor,versionMinor,team_char_1,team_char_2,teams_chars,scoreRecover,scoreProtect,scoreVision,scoreCreditRevive,_bestWeapon_code,bestWeapon_kr,weapon_games,character_total_games,weapon_pick_rate_within_character,weapon_win_rate,equip_combo_kr_top1,equip_combo_kr_top2,equip_combo_kr_top3,games_top1,games_top2,games_top3,win_rate_top1,win_rate_top2,win_rate_top3
0,펠릭스,35260427,49,19,5,6,1,3,1209,60,49,67,39,90,62,0,27,0,0,100,0,1082,150,0,800,0,22,110,21,960,35,50,0.13,2.60,rank_game,22,1,마이,이안,마이이안,0.000000,0.099825,0.280000,0.000000,19,창,2548,2548,1.000000,0.136578,애각창 | 빛의 증표 | 고스트 | 큐브 워치 | 분홍신,애각창 | 빛의 증표 | 고스트 | 큐브 워치 | 미스릴 부츠,애각창 | 빛의 증표 | 타이탄 아머 | 큐브 워치 | 레이싱 부츠,20.0,120.0,20.0,0.450000,0.308333,0.200000
1,수아,35260427,28,13,4,2,6,2,1265,20,32,75,33,36,44,0,47,0,0,150,0,998,120,0,600,0,168,110,27,1070,39,57,0.12,1.90,rank_game,22,1,아이솔,유키,아이솔유키,0.000000,0.067151,0.360000,0.166667,13,망치,1780,2218,0.802525,0.141573,개밥바라기 | 예언자의 터번 | 성법의 | 미스릴 부츠 | 임세티,개밥바라기 | 예언자의 터번 | 성법의 | 용의 비늘 | 미스릴 부츠,개밥바라기 | 예언자의 터번 | 성법의 | 용의 비늘 | 경량화 부츠,48.0,33.0,20.0,0.479167,0.333333,0.250000
2,캐시,35260427,23,15,2,2,8,3,1068,20,35,39,10,58,40,0,15,0,3,40,0,780,300,0,370,0,0,110,18,940,29,53,0.09,1.65,rank_game,22,1,매그너스,에키온,매그너스에키온,0.026183,0.152215,0.240000,0.000000,15,단검,1043,3400,0.306765,0.182167,환영도 | 우주 비행사의 헬멧 | 레버넌트 | 아흐라만의 손길 | 헤르메스의 부츠,환영도-새벽 | 우주 비행사의 헬멧 | 레버넌트 | 아흐라만의 손길 | 헤르메스의 부츠,월식 | 검은 죽음 | 성법의 | 틴달로스의 군주 | 칼날 다리,20.0,89.0,27.0,0.850000,0.730337,0.518519
3,마커스,35260427,53,14,1,4,3,0,1302,40,52,49,28,58,36,3,22,3,0,100,0,1130,420,0,400,0,200,110,31,920,34,50,0.12,2.41,rank_game,22,1,데비&마를렌,바냐,데비&마를렌바냐,0.000000,0.085807,0.413333,0.166667,14,도끼,3109,3425,0.907737,0.151817,반고부 | 불꽃 드레스 | 미스릴 방패 | 와일드 워커 | 택티컬 바이저,반고부 | 미스릴 갑옷 | 미스릴 부츠 | 택티컬 바이저 | 토템,반고부 | 타이탄 아머 | 와일드 워커 | 택티컬 바이저 | 토템,41.0,43.0,112.0,0.609756,0.534884,0.357143
4,데비&마를렌,35260427,65,16,1,2,4,2,1186,20,50,49,28,62,38,5,23,3,0,0,0,1120,360,0,650,0,0,110,25,970,33,52,0.12,4.00,rank_game,22,1,마커스,바냐,마커스바냐,0.000000,0.000000,0.333333,0.000000,16,양손검,7560,7560,1.000000,0.139815,레바테인 | 블래스터 헬멧 | 유령 신부의 드레스 | 미라지 워커 | 토템,레바테인 | 변검 | 유령 신부의 드레스 | 블레이드 부츠 | 달빛 펜던트,레바테인 | 미스릴 갑옷 | 분홍신 | 달빛 펜던트 | 택티컬 바이저,36.0,20.0,53.0,0.611111,0.600000,0.566038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86137,다르코,35813230,74,3,8,0,0,1,159,0,5,10,0,4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,960,42,48,0.12,1.85,rank_game,23,0,아야,에이든,아야에이든,0.000000,0.000000,0.000000,0.000000,3,방망이,7422,7422,1.000000,0.139181,로드 오브 하트-진홍 | 레이싱 슈트 | 갤럭시 스텝 | 택티컬 바이저 | 토템,여의봉 | 유령 신부의 드레스 | 헬릭스 | 블레이드 부츠 | 택티컬 바이저,여의봉 | 레이싱 슈트 | 오토-암즈 | 갤럭시 스텝 | 택티컬 바이저,25.0,92.0,28.0,0.600000,0.554348,0.535714
86138,테오도르,35827539,62,11,8,0,0,1,120,0,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,940,45,53,0.10,6.43,rank_game,23,0,유키,재키,유키재키,0.000000,0.000000,0.000000,0.000000,11,저격총,1800,1800,1.000000,0.186667,위도우 메이커-새벽 | 레이싱 헬멧 | 성법의 | 텔루리안 타임피스 | 블레이드 부츠,사사성광 | 레이싱 헬멧 | 성법의 | 텔루리안 타임피스 | 분홍신,사사성광 | 레이싱 헬멧 | 성법의 | 텔루리안 타임피스 | 블레이드 부츠,34.0,25.0,132.0,0.647059,0.600000,0.446970
86139,아야,35551568,2,9,3,0,0,1,659,0,36,27,28,34,44,0,14,0,0,0,0,0,0,0,0,0,0,0,0,905,43,51,0.11,5.25

In [26]:

### 추가된 컬럼 ###
## 랭크게임 필터링이 언급되지 않는 컬럼들은 태블로에서 걸어줘야함 ## 

# 0-1. team_char_1 : 팀원1 - 랭크 필터링 필요없음
# 0-2. team_char_2 : 팀원2 - 랭크 필터링 필요없음
# 0-3. team_chars  : 팀원들 - 랭크 필터링 필요없음


#  기존 수치들을 0~1로 스케일링한 컬럼(대시보드용)
#  1. scoreRecover : teamRecover 정규화(팀 회복 기여도 스코어)
#  2. scoreProtect : protectAbsorb 정규화(보호막/흡수 스코어)
#  3. scoreVision : viewContribution 정규화(시야 기여 스코어)
#  4. scoreCreditRevive : creditRevivedOthersCount 정규화(아군 부활 기여 스코어)


#  B) 주무기(무기) 매핑/통계 컬럼 (보통 5~6개)
#  캐릭터-주무기 기준으로 픽률/승률 붙이기 위한 것들
#  5. _bestWeapon_code : bestWeapon을 숫자(Int64)로 강제 변환한 “무기코드”(merge 키 안정화용)
#  6. bestWeapon_kr : _bestWeapon_code를 weapon_type 딕셔너리로 매핑한 “무기 한글명”
#  7. weapon_games : 해당 캐릭터가 해당 주무기를 든 경기 수
#  8. character_total_games : 해당 캐릭터의 전체 경기 수
#  9. weapon_pick_rate_within_character : 캐릭터 내부 주무기 픽률 = weapon_games / character_total_games
#  10. weapon_win_rate : 해당 캐릭터가 해당 무기를 들었을 때 승률(0~1)


#  C) 캐릭터별 장비조합 승률 Top3 (wide 컬럼 9개)
#  Top1
#  11. equip_combo_kr_top1 : 1위 장비조합(한글 조합 문자열)
#  12. win_rate_top1 : 1위 조합 승률(0~1)
#  13. games_top1 : 1위 조합 표본 수(게임 수)

#  Top2
#  14. equip_combo_kr_top2
#  15. win_rate_top2
#  16. games_top2

#  Top3
#  17. equip_combo_kr_top3
#  18. win_rate_top3
#  19. games_top3



In [27]:
df_teab['weapon_games'].sort_values(ascending = False)

30050    7560
53307    7560
40264    7560
63463    7560
27482    7560
         ... 
83903     156
64643     156
70904     156
36209     156
78618     156
Name: weapon_games, Length: 86142, dtype: int64

In [28]:
df_teab['viewContribution']

0        21
1        27
2        18
3        31
4        25
         ..
86137     0
86138     0
86139     0
86140     0
86141     0
Name: viewContribution, Length: 86142, dtype: int64

In [33]:
df_teab['weapon_games'].value_counts(ascending=False)

weapon_games
3839    3766
7560    3533
5975    2816
6507    2809
7112    2791
        ... 
193       89
359       87
316       86
171       58
156       47
Name: count, Length: 97, dtype: int64